# v11_cpe_lc_v2 — Ineligible Combo Gating Analysis

**Purpose:** Quantify the gating rate and assess prediction quality for *ineligible* `(target_game_id, sdk_event_name)` combos — those the model zeroes cost for because they were never seen during training.

**Gating mechanism (`DeployModel.forward`):**
```
gate  = _gate_tensor[tgid_idx, sdk_idx]           # 1.0 = seen in training, 0.0 = unseen
p     = psn_pred * calib_factor                   # prediction is NOT gated
cost  = max_cost × discount_factor × p × gate     # cost IS zeroed for unseen combos
```

**Gating signal in `prediction_v1alpha1`:** `p > 0` AND `dcpi = 0` AND `max_cost > 0`

**Table structure:** `app_event_campaigns` is a STRUCT with parallel REPEATED arrays (one entry per campaign scored in the auction). Accessed via `UNNEST ... WITH OFFSET`.

| Field | Meaning |
|---|---|
| `app_event_campaigns.campaign_ids[i]` | Campaign ID |
| `app_event_campaigns.p[i]` | Model prediction (NOT gated) |
| `app_event_campaigns.dcpi[i]` | Actual bid in microdollars (0 if gated) |
| `app_event_campaigns.max_cost[i]` | Advertiser target CPE in microdollars |
| `app_event_campaigns.sdk_event_name[i]` | Targeted SDK event name |
| `app_event_campaigns.app_event_type[i]` | `APP_EVENT_CONVERSION_TYPE_LEVEL_COMPLETE` |
| `_lapio_submit_time` | Partition key (HOUR granularity, 30-day retention) |

**Sections:**
1. Overall Gating Rate (Approach 1)
2. Gating Rate by Platform
3. Top Gated `(campaign_id, sdk_event_name)` Combos (Approach 2)
4. Prediction Quality on Ineligible Combos — p-value distribution
5. Spend Opportunity Cost
6. Summary

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from google.cloud import bigquery
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

client = bigquery.Client(project='unity-ads-ds-prd')

# ---------------------------------------------------------------------------
# Table constants
# ---------------------------------------------------------------------------
PRED_TABLE  = 'unity-data-ads-prd.dcpi.prediction_v1alpha1'
CAMPS_TABLE = 'unity-data-ads-core-prd.ads_dimension_data.campaigns_v3'

# LC event type enum in prediction_v1alpha1
LC_TYPE = 'APP_EVENT_CONVERSION_TYPE_LEVEL_COMPLETE'

# ---------------------------------------------------------------------------
# Date range — 7 days  (table expires after 30d; HOUR partition + UNNEST
# expansion makes longer windows very expensive)
# ---------------------------------------------------------------------------
ANALYSIS_START = '2026-08-25'
ANALYSIS_END   = '2026-08-31'
TS_START = f'{ANALYSIS_START} 00:00:00'
TS_END   = f'{ANALYSIS_END} 23:59:59'

# ---------------------------------------------------------------------------
# Efficient CTE pattern: GENERATE_ARRAY + SAFE_OFFSET
# Avoids 5-way UNNEST WITH OFFSET joins; BQ can filter on type before
# expanding the array index.
# ---------------------------------------------------------------------------
LC_CTE = f"""
lc_campaigns AS (
  SELECT
    DATE(_lapio_submit_time)                              AS submit_date,
    app_event_campaigns.campaign_ids[SAFE_OFFSET(i)]     AS campaign_id,
    app_event_campaigns.p[SAFE_OFFSET(i)]                AS p_val,
    CAST(app_event_campaigns.dcpi[SAFE_OFFSET(i)]
         AS FLOAT64)                                     AS dcpi,
    CAST(app_event_campaigns.max_cost[SAFE_OFFSET(i)]
         AS FLOAT64)                                     AS max_cost,
    app_event_campaigns.sdk_event_name[SAFE_OFFSET(i)]   AS sdk_event_name
  FROM `{PRED_TABLE}`,
    UNNEST(GENERATE_ARRAY(
      0,
      ARRAY_LENGTH(app_event_campaigns.campaign_ids) - 1
    )) AS i
  WHERE _lapio_submit_time BETWEEN '{TS_START}' AND '{TS_END}'
    AND app_event_campaigns.app_event_type[SAFE_OFFSET(i)] = '{LC_TYPE}'
)"""

def run_query(sql: str) -> pd.DataFrame:
    return client.query(sql).to_dataframe()

print(f'Analysis window : {ANALYSIS_START} → {ANALYSIS_END}')
print(f'Prediction table: {PRED_TABLE}')


---
## 1. Overall Gating Rate (Approach 1)

Classify every LC campaign-auction row as:
- **Eligible** — `p > 0` AND `dcpi > 0`  — model bid
- **Gated** — `p > 0` AND `dcpi = 0` AND `max_cost > 0` — model scored, gate zeroed bid
- **Zero-p** — `p = 0` — model predicted zero (not the gate)

Gating rate = gated / (gated + eligible)

In [ ]:
sql_gating_overall = f'''
WITH {LC_CTE}
SELECT
  submit_date,
  COUNT(*)                                                       AS total_requests,
  COUNTIF(p_val > 0 AND dcpi > 0)                               AS eligible_requests,
  COUNTIF(p_val > 0 AND dcpi = 0 AND max_cost > 0)              AS gated_requests,
  COUNTIF(p_val = 0)                                            AS zero_p_requests,
  ROUND(SAFE_DIVIDE(
    COUNTIF(p_val > 0 AND dcpi = 0 AND max_cost > 0),
    COUNTIF(p_val > 0 AND max_cost > 0)
  ) * 100, 2)                                                    AS gating_rate_pct,
  ROUND(AVG(IF(p_val > 0 AND dcpi = 0 AND max_cost > 0, p_val, NULL)), 6)
                                                                 AS gated_avg_pred,
  ROUND(AVG(IF(p_val > 0 AND dcpi > 0, p_val, NULL)), 6)        AS eligible_avg_pred
FROM lc_campaigns
GROUP BY submit_date
ORDER BY submit_date
'''

df_gating = run_query(sql_gating_overall)

total    = df_gating['total_requests'].sum()
eligible = df_gating['eligible_requests'].sum()
gated    = df_gating['gated_requests'].sum()
zero_p   = df_gating['zero_p_requests'].sum()
gate_rate = gated / (gated + eligible) * 100 if (gated + eligible) > 0 else 0

print('='*60)
print(f'OVERALL GATING SUMMARY  ({ANALYSIS_START} → {ANALYSIS_END})')
print('='*60)
print(f'  Total LC requests      : {total:>15,}')
print(f'  Eligible (dcpi > 0)    : {eligible:>15,}  ({eligible/total*100:.1f}%)')
print(f'  Gated (p>0, dcpi=0)    : {gated:>15,}  ({gated/total*100:.1f}%)')
print(f'  Zero-p (p=0)           : {zero_p:>15,}  ({zero_p/total*100:.1f}%)')
print(f'  Gating rate (of p>0)   : {gate_rate:>14.2f}%')
print('='*60)
display(df_gating)


In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=df_gating['submit_date'], y=df_gating['eligible_requests'],
    name='Eligible', marker_color='#2e86c1', opacity=0.7, yaxis='y2',
))
fig.add_trace(go.Bar(
    x=df_gating['submit_date'], y=df_gating['gated_requests'],
    name='Gated', marker_color='#e8302a', opacity=0.7, yaxis='y2',
))
fig.add_trace(go.Scatter(
    x=df_gating['submit_date'], y=df_gating['gating_rate_pct'],
    mode='lines+markers', name='Gating Rate (%)',
    line=dict(color='#c0392b', width=2.5), yaxis='y',
))
fig.update_layout(
    title='Daily Gating Rate — v11_cpe_lc_v2 LC Requests',
    barmode='stack', template='plotly_white', height=480,
    yaxis=dict(title='Gating Rate (%)', side='left', showgrid=True),
    yaxis2=dict(title='Request Count', side='right', overlaying='y', showgrid=False),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

In [ ]:
# avg(p) for gated vs eligible — if gated avg_pred is close to eligible,
# the model is confident about suppressed combos (potential false negatives).
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_gating['submit_date'], y=df_gating['gated_avg_pred'],
    mode='lines+markers', name='Gated avg(p)',
    line=dict(color='#e8302a', width=2),
))
fig.add_trace(go.Scatter(
    x=df_gating['submit_date'], y=df_gating['eligible_avg_pred'],
    mode='lines+markers', name='Eligible avg(p)',
    line=dict(color='#2e86c1', width=2),
))
fig.update_layout(
    title='Average Model Prediction (p) — Gated vs Eligible',
    xaxis_title='Date', yaxis_title='avg(p)',
    template='plotly_white', height=420,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

---
## 2. Gating Rate by Platform

In [ ]:
sql_model_version = f'''
WITH {LC_CTE},
model_ver AS (
  SELECT
    (SELECT v.value FROM UNNEST(app_event_model_version) v
     WHERE v.key = 'level_complete' LIMIT 1)  AS lc_model_version,
    app_event_campaigns.p[SAFE_OFFSET(i)]      AS p_val,
    CAST(app_event_campaigns.dcpi[SAFE_OFFSET(i)] AS FLOAT64) AS dcpi,
    CAST(app_event_campaigns.max_cost[SAFE_OFFSET(i)] AS FLOAT64) AS max_cost
  FROM `{PRED_TABLE}`,
    UNNEST(GENERATE_ARRAY(0, ARRAY_LENGTH(app_event_campaigns.campaign_ids)-1)) AS i
  WHERE _lapio_submit_time BETWEEN '{TS_START}' AND '{TS_END}'
    AND app_event_campaigns.app_event_type[SAFE_OFFSET(i)] = '{LC_TYPE}'
)
SELECT
  lc_model_version,
  COUNT(*)                                                      AS total_requests,
  COUNTIF(p_val > 0 AND dcpi > 0)                              AS eligible,
  COUNTIF(p_val > 0 AND dcpi = 0 AND max_cost > 0)             AS gated,
  ROUND(SAFE_DIVIDE(
    COUNTIF(p_val > 0 AND dcpi = 0 AND max_cost > 0),
    COUNTIF(p_val > 0 AND max_cost > 0)
  ) * 100, 2)                                                   AS gating_rate_pct
FROM model_ver
GROUP BY lc_model_version
ORDER BY total_requests DESC
'''

df_version = run_query(sql_model_version)
print('Gating rate by model version (confirms v11_cpe_lc_v2 traffic split):')
display(df_version)


---
## 3. Top Gated `(campaign_id, sdk_event_name)` Combos (Approach 2)

`sdk_event_name` is available directly in `app_event_campaigns` — no campaign join needed.
Join with `campaigns_v3` to add `game_id` for the `(game_id, sdk_event_name)` gate key.

In [ ]:
sql_top_gated_combos = f'''
WITH {LC_CTE},
camps AS (
  SELECT id AS campaign_id, game_id
  FROM `{CAMPS_TABLE}`
  WHERE app_event_conversion_type = 'LEVEL_COMPLETE'
    AND archived_at IS NULL
)
SELECT
  c.game_id                                               AS target_game_id,
  lc.sdk_event_name,
  COUNT(DISTINCT lc.campaign_id)                          AS campaign_count,
  COUNT(*)                                                AS total_requests,
  COUNTIF(lc.p_val > 0 AND lc.dcpi = 0 AND lc.max_cost > 0)
                                                          AS gated_requests,
  COUNTIF(lc.p_val > 0 AND lc.dcpi > 0)                  AS eligible_requests,
  ROUND(SAFE_DIVIDE(
    COUNTIF(lc.p_val > 0 AND lc.dcpi = 0 AND lc.max_cost > 0),
    COUNT(*)
  ) * 100, 2)                                             AS gating_rate_pct,
  ROUND(AVG(IF(lc.p_val > 0 AND lc.dcpi = 0 AND lc.max_cost > 0, lc.p_val, NULL)), 6)
                                                          AS gated_avg_pred,
  ROUND(AVG(IF(lc.p_val > 0 AND lc.dcpi > 0, lc.p_val, NULL)), 6)
                                                          AS eligible_avg_pred,
  -- Upper-bound suppressed spend: max_cost * p (microdollars → USD)
  ROUND(SUM(IF(lc.p_val > 0 AND lc.dcpi = 0 AND lc.max_cost > 0,
               lc.max_cost * lc.p_val / 1e6, 0)), 2)    AS est_suppressed_spend_usd
FROM lc_campaigns lc
LEFT JOIN camps c ON c.campaign_id = lc.campaign_id
GROUP BY c.game_id, lc.sdk_event_name
HAVING COUNTIF(lc.p_val > 0 AND lc.dcpi = 0 AND lc.max_cost > 0) > 0
ORDER BY gated_requests DESC
LIMIT 100
'''

df_combos = run_query(sql_top_gated_combos)
print(f'Distinct gated (game_id, sdk_event_name) pairs: {len(df_combos)}')
display(df_combos.head(30))


In [ ]:
top_n = 25
df_top = df_combos.head(top_n).copy()
df_top['label'] = df_top['target_game_id'].fillna('?').astype(str) + '  /  ' + df_top['sdk_event_name'].astype(str)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=df_top['label'], x=df_top['gated_requests'],
    name='Gated', orientation='h', marker_color='#e8302a', opacity=0.85,
    customdata=df_top[['gating_rate_pct', 'gated_avg_pred', 'est_suppressed_spend_usd']].values,
    hovertemplate=('%{y}<br>gated: %{x:,}<br>rate: %{customdata[0]:.1f}%'
                   '<br>avg p: %{customdata[1]:.5f}<br>suppressed spend (UB): $%{customdata[2]:,.0f}<extra></extra>'),
))
fig.add_trace(go.Bar(
    y=df_top['label'], x=df_top['eligible_requests'],
    name='Eligible', orientation='h', marker_color='#2e86c1', opacity=0.35,
))
fig.update_layout(
    title=f'Top {top_n} Gated (game_id, sdk_event_name) Combos by Gated Volume',
    barmode='stack', xaxis_title='Request Count',
    template='plotly_white',
    height=max(500, top_n * 28 + 180),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

In [ ]:
# Scatter: gated volume vs model confidence
# Top-right = high volume + high model confidence = best candidates for gate expansion
df_s = df_combos[df_combos['gated_requests'] >= 100].copy()
df_s['label'] = df_s['target_game_id'].fillna('?').astype(str) + ' / ' + df_s['sdk_event_name']
eligible_global_median_p = df_combos['eligible_avg_pred'].median()

fig = px.scatter(
    df_s, x='gated_requests', y='gated_avg_pred',
    size='est_suppressed_spend_usd',
    color='gating_rate_pct',
    hover_name='label',
    hover_data=['target_game_id', 'sdk_event_name',
                'gating_rate_pct', 'est_suppressed_spend_usd', 'campaign_count'],
    color_continuous_scale='Reds',
    title='Gated Combos: Volume vs Model Confidence (bubble = suppressed spend UB)',
    labels={'gated_requests': 'Gated Requests', 'gated_avg_pred': 'Avg Model Prediction p'},
    template='plotly_white', height=550,
)
fig.add_hline(
    y=eligible_global_median_p, line_dash='dot', line_color='#2e86c1',
    annotation_text=f'Eligible median avg(p) = {eligible_global_median_p:.5f}',
    annotation_position='right',
)
fig.show()
print(f'Combos with gated_avg_pred > eligible median ({eligible_global_median_p:.5f}):')
high_conf = df_s[df_s['gated_avg_pred'] > eligible_global_median_p].sort_values('est_suppressed_spend_usd', ascending=False)
print(f'  Count: {len(high_conf)}')
display(high_conf[['target_game_id','sdk_event_name','gated_requests',
                    'gating_rate_pct','gated_avg_pred','est_suppressed_spend_usd']].head(20))

In [ ]:
# Games with BOTH eligible and gated events → LC-active games with expansion potential
game_summary = (
    df_combos.groupby('target_game_id')
    .apply(lambda g: pd.Series({
        'eligible_combos':     int((g['eligible_requests'] > 0).sum()),
        'gated_combos':        int((g['gated_requests']   > 0).sum()),
        'total_gated_requests': int(g['gated_requests'].sum()),
        'suppressed_spend_usd': float(g['est_suppressed_spend_usd'].sum()),
        'avg_gated_pred':       float(g.loc[g['gated_requests']   > 0, 'gated_avg_pred'].mean()),
        'avg_eligible_pred':    float(g.loc[g['eligible_requests']> 0, 'eligible_avg_pred'].mean()),
    }))
    .reset_index()
)
mixed = game_summary[
    (game_summary['eligible_combos'] > 0) & (game_summary['gated_combos'] > 0)
].sort_values('total_gated_requests', ascending=False)

print(f'Games with BOTH eligible and gated sdk_event_name combos: {len(mixed)}')
print('LC-active games — gated events most likely to benefit from gate expansion.')
display(mixed.head(30))

---
## 4. Prediction Quality on Ineligible Combos

Since gated requests have `dcpi = 0` (no bid, no win, no install), outcome-based bias (pred/actual − 1) is not computable.  
We assess prediction quality via:

1. **P-value distribution** — NTILE(20) for gated vs eligible.  
   Gated avg(p) ≈ eligible avg(p) → model treats them similarly; gate may suppress valid traffic.  
   Gated avg(p) ≪ eligible avg(p) → model already discounts these combos; gate is aligned.

2. **High-confidence gated combos** — those where gated avg(p) exceeds the eligible median.

In [ ]:
sql_pdist = f'''
WITH {LC_CTE},
ranked AS (
  SELECT
    p_val,
    (p_val > 0 AND dcpi = 0 AND max_cost > 0)  AS is_gated,
    NTILE(20) OVER (
      PARTITION BY (p_val > 0 AND dcpi = 0 AND max_cost > 0)
      ORDER BY p_val
    ) AS p_bucket
  FROM lc_campaigns
  WHERE p_val > 0 AND max_cost > 0
)
SELECT
  IF(is_gated, 'gated', 'eligible')  AS cohort,
  p_bucket,
  ROUND(MIN(p_val), 7)               AS p_min,
  ROUND(MAX(p_val), 7)               AS p_max,
  ROUND(AVG(p_val), 7)               AS p_avg,
  COUNT(*)                           AS request_count
FROM ranked
GROUP BY cohort, p_bucket
ORDER BY cohort, p_bucket
'''

df_pdist = run_query(sql_pdist)
pivot = df_pdist.pivot(index='p_bucket', columns='cohort', values='p_avg').round(7)
print('Avg p per NTILE bucket — gated vs eligible:')
display(pivot)


In [ ]:
gated_df    = df_pdist[df_pdist['cohort'] == 'gated'].sort_values('p_bucket')
eligible_df = df_pdist[df_pdist['cohort'] == 'eligible'].sort_values('p_bucket')

fig = go.Figure()
fig.add_trace(go.Bar(
    x=gated_df['p_bucket'], y=gated_df['p_avg'],
    name='Gated avg(p)', marker_color='#e8302a', opacity=0.85,
    customdata=gated_df[['p_min', 'p_max', 'request_count']].values,
    hovertemplate='Bucket %{x}<br>avg p: %{y:.7f}<br>range: [%{customdata[0]:.7f}, %{customdata[1]:.7f}]<br>count: %{customdata[2]:,}<extra></extra>',
))
fig.add_trace(go.Bar(
    x=eligible_df['p_bucket'], y=eligible_df['p_avg'],
    name='Eligible avg(p)', marker_color='#2e86c1', opacity=0.85,
    customdata=eligible_df[['p_min', 'p_max', 'request_count']].values,
    hovertemplate='Bucket %{x}<br>avg p: %{y:.7f}<br>range: [%{customdata[0]:.7f}, %{customdata[1]:.7f}]<br>count: %{customdata[2]:,}<extra></extra>',
))
fig.update_layout(
    title='P-Value Distribution: Gated vs Eligible (NTILE 20)',
    barmode='group',
    xaxis_title='P-Value Bucket (1=lowest, 20=highest)',
    yaxis_title='Average p',
    template='plotly_white', height=460,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

g_med = gated_df['p_avg'].median()
e_med = eligible_df['p_avg'].median()
print(f'Median bucket avg(p) — gated: {g_med:.6f}  |  eligible: {e_med:.6f}  |  ratio: {g_med/e_med:.2f}x')

---
## 5. Spend Opportunity Cost

**Upper-bound formula (per campaign-auction row):**
```
suppressed_spend ≈ max_cost × p   (microdollars)
```
Ignores `discount_factor` and win rate — actual would be lower, but the suppression ratio vs actual spend is informative.

In [ ]:
sql_spend = f'''
WITH {LC_CTE}
SELECT
  submit_date,
  ROUND(SUM(IF(dcpi > 0, dcpi / 1e6, 0)), 2)                        AS actual_spend_usd,
  ROUND(SUM(IF(p_val > 0 AND dcpi = 0 AND max_cost > 0,
               max_cost * p_val / 1e6, 0)), 2)                       AS suppressed_spend_ub_usd,
  ROUND(SAFE_DIVIDE(
    SUM(IF(p_val > 0 AND dcpi = 0 AND max_cost > 0, max_cost * p_val / 1e6, 0)),
    SUM(IF(dcpi > 0, dcpi / 1e6, 0))
  ), 4)                                                               AS suppression_ratio
FROM lc_campaigns
GROUP BY submit_date
ORDER BY submit_date
'''

df_spend = run_query(sql_spend)

total_actual     = df_spend['actual_spend_usd'].sum()
total_suppressed = df_spend['suppressed_spend_ub_usd'].sum()
ratio            = total_suppressed / total_actual if total_actual > 0 else 0

print('='*60)
print(f'SPEND OPPORTUNITY COST  ({ANALYSIS_START} → {ANALYSIS_END})')
print('='*60)
print(f'  Actual spend             : ${total_actual:>14,.2f}')
print(f'  Suppressed spend (UB)    : ${total_suppressed:>14,.2f}')
print(f'  Suppression ratio        : {ratio:>14.2f}x')
print('  (UB = upper bound; ignores discount_factor + win rate)')
print('='*60)
display(df_spend)


In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=df_spend['submit_date'], y=df_spend['actual_spend_usd'],
    name='Actual Spend ($)', marker_color='#2e86c1', opacity=0.85,
))
fig.add_trace(go.Bar(
    x=df_spend['submit_date'], y=df_spend['suppressed_spend_ub_usd'],
    name='Suppressed (UB) ($)', marker_color='#e8302a', opacity=0.60,
))
fig.add_trace(go.Scatter(
    x=df_spend['submit_date'], y=df_spend['suppression_ratio'],
    mode='lines+markers', name='Suppression Ratio (x)',
    line=dict(color='#e8a020', width=2), yaxis='y2',
))
fig.update_layout(
    title='Daily Actual vs Gate-Suppressed Spend — v11_cpe_lc_v2',
    barmode='group',
    yaxis=dict(title='Spend (USD)', side='left'),
    yaxis2=dict(title='Suppression Ratio (x)', side='right', overlaying='y', showgrid=False),
    template='plotly_white', height=460,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

In [ ]:
top20 = df_combos.nlargest(20, 'est_suppressed_spend_usd').copy()
top20['label'] = top20['target_game_id'].fillna('?').astype(str) + '  /  ' + top20['sdk_event_name']

fig = go.Figure(go.Bar(
    y=top20['label'], x=top20['est_suppressed_spend_usd'],
    orientation='h', marker_color='#e8302a', opacity=0.85,
    customdata=top20[['gated_requests', 'gating_rate_pct', 'gated_avg_pred']].values,
    hovertemplate=('%{y}<br>suppressed spend (UB): $%{x:,.2f}'
                   '<br>gated requests: %{customdata[0]:,}'
                   '<br>gating rate: %{customdata[1]:.1f}%'
                   '<br>avg p: %{customdata[2]:.5f}'
                   '<extra></extra>'),
))
fig.update_layout(
    title='Top 20 Gated Combos by Estimated Suppressed Spend (UB)',
    xaxis_title='Suppressed Spend (USD)',
    template='plotly_white',
    height=max(400, 20 * 26 + 180),
)
fig.show()

---
## 6. Summary

In [ ]:
_total    = df_gating['total_requests'].sum()
_eligible = df_gating['eligible_requests'].sum()
_gated    = df_gating['gated_requests'].sum()
_gate_pct = _gated / (_gated + _eligible) * 100 if (_gated + _eligible) > 0 else 0
_actual_spend     = df_spend['actual_spend_usd'].sum()
_suppressed_spend = df_spend['suppressed_spend_ub_usd'].sum()
_suppression_ratio = _suppressed_spend / _actual_spend if _actual_spend > 0 else 0
_g_med = gated_df['p_avg'].median()
_e_med = eligible_df['p_avg'].median()
_num_combos     = len(df_combos)
_high_conf_n    = len(high_conf) if 'high_conf' in dir() else 0
_mixed_games_n  = len(mixed)

print('=' * 70)
print(f'  INELIGIBLE COMBO GATING REPORT — v11_cpe_lc_v2')
print(f'  Period: {ANALYSIS_START} → {ANALYSIS_END}')
print('=' * 70)

print('\n── Approach 1: Gating Rate ───────────────────────────────────────')
print(f'  Total LC requests              : {_total:>15,}')
print(f'  Eligible (dcpi > 0)            : {_eligible:>15,}  ({_eligible/_total*100:.1f}%)')
print(f'  Gated (p>0, dcpi=0)            : {_gated:>15,}  ({_gated/_total*100:.1f}%)')
print(f'  Gating rate (of p>0 requests)  : {_gate_pct:>14.2f}%')

print('\n── Approach 2: Combo Coverage ────────────────────────────────────')
print(f'  Distinct gated (game, event) pairs : {_num_combos:>10,}')
print(f'  LC-active games with gated events  : {_mixed_games_n:>10,}')

print('\n── Prediction Quality ────────────────────────────────────────────')
print(f'  Median avg(p) — gated    : {_g_med:.6f}')
print(f'  Median avg(p) — eligible : {_e_med:.6f}')
print(f'  Ratio (gated / eligible) : {_g_med / _e_med:.2f}x')
print(f'  High-conf gated combos*  : {_high_conf_n}  (* avg_p > eligible median, >= 100 req.)')

print('\n── Spend Opportunity Cost ────────────────────────────────────────')
print(f'  Actual spend             : ${_actual_spend:>14,.2f}')
print(f'  Suppressed spend (UB)    : ${_suppressed_spend:>14,.2f}')
print(f'  Suppression ratio        : {_suppression_ratio:>14.2f}x')

print('\n── Interpretation ────────────────────────────────────────────────')
if _gate_pct > 30:
    print(f'  ⚠ HIGH gating rate ({_gate_pct:.1f}%). Review trained_game_sdk_combo.json.')
elif _gate_pct > 10:
    print(f'  ⚡ Moderate gating rate ({_gate_pct:.1f}%). Monitor for growth.')
else:
    print(f'  ✅ Low gating rate ({_gate_pct:.1f}%). Gate covers most live traffic.')

if _g_med / _e_med > 0.5:
    print(f'  ⚠ Gated median p = {_g_med/_e_med:.0%} of eligible — model is moderately confident')
    print(f'    about ineligible combos. Consider expanding training coverage.')
else:
    print(f'  ✅ Gated avg(p) well below eligible — gate is aligned with model confidence.')
print('=' * 70)